## State estimation

The purpose of state estimation is to reduce the uncertainty of the physical world to which our robot is subjected. During its treasure hunt, our Thymio robot evolves by detecting its environment and performing actions based on mathematical representations of that environment. Although this environment appears relatively straightforward, it is inherently unpredictable. This is partly because the robot can only perceive it through imperfect sensors that are difficult to model due to the complexity of their manufacturing processes. However, the environment is also unpredictable due to its variability over time.

Therefore, we need a model that can filter out this uncertainty, enabling our robot to perform its tasks with greater robustness. However, in order to select such a model, we must first define the state-space representation of our system.

### Treasure map recall

<div style="text-align: center;">
    <img src="./imgs/map_drawing.jpeg" alt="Treasure map" width="800"/>
</div>

### State-space representation

#### State variable definition

As we have seen, the state variable is the smallest possible subset of variables required to describe the evolution of the robot over time. It means that it should contain only the important quantities that are affecting the robot state and no redundant informations. For this project, we'll define the state of the Thymio as follow:

- Thymio is a differential dirve robot (DDR) with 3 DOFs, so three variables are required to describe its pose: $x$, $y$ and $\theta$

- Given that the robot is moving at a constant linear and/or angular velocity, its pose at the next time step can be computed directly from its previous pose and the measured speeds of its motors. Therefore, we can expressed its motion model in its referential frame as follow:

```math
\begin{equation*}
    \begin{align*}
        x_{k} &= x_{k-1} + \Delta T v \\
        \theta_{k} &= \theta_{k-1} + \Delta T \omega
    \end{align*}
\end{equation*}
```

- As explained in [1-Components of a Mobile Robot, slide 11](https://moodle.epfl.ch/pluginfile.php/2703472/mod_resource/content/13/Slides%2001%20-%20Components%20of%20a%20Mobile%20Robot.pdf), the forward and angular speeds are related to the motor speeds with the following equations:

```math
\begin{equation*}
    \begin{align*}
        v &= \frac{r\dot{\phi}_r}{2} + \frac{r\dot{\phi}_l}{2} \\
        \omega &= \frac{r\dot{\phi}_r}{d}-\frac{r\dot{\phi}_l}{d}
    \end{align*}
\end{equation*}
```

Where $\dot{\phi}_i$ is the motor speed, $r$ the wheel radius and $d$ the distance between both wheels.

- Although the motor speeds $\dot{\phi}_r$ and $\dot{\phi}_l$ appear in the motion model, they are not quantities that the filter must estimate: they are directly provided by the robot’s speed sensors (odometry) or motor commands. Once these speeds are used to predict the position, they are no longer needed as part of the state. Including them as states would add dimensions to the covariance matrix and increase computational complexity without providing additional useful information.

Hence, the complete state variable will be denoted as $\vec{x}(t) =
\begin{pmatrix}
  x\\ 
  y\\
  \theta
\end{pmatrix}$.

#### State-space equations

Expressing the Thymio’s state in a fixed reference frame relative to the map leads to a nonlinear system, because the robot’s velocity must be projected onto the reference axes using trigonometric (sine and cosine) terms. To map the velocities from the robot's referential frame to the global reference frame of the map, we use rotational matrices as we have seen in [1-Components of a Mobile Robot, slide 12](https://moodle.epfl.ch/pluginfile.php/2703472/mod_resource/content/13/Slides%2001%20-%20Components%20of%20a%20Mobile%20Robot.pdf):

- To align both x axis, we perform a rotation around the z axis of the global frame, using:

```math
\begin{equation*}
    R_z(-\theta) = R_z(\theta)^{-1} =
    \begin{pmatrix}
        cos(\theta) & sin(\theta) & 0 \\
        -sin(\theta) & cos(\theta) & 0 \\
        0 & 0 & 1
    \end{pmatrix}
\end{equation*}
```

- Then to reverse the z axis, we perform a rotation around the x axis of the global frame, using:

```math
\begin{equation*}
    R_x(\pi) =
    \begin{pmatrix}
        1 & 0 & 0 \\
        0 & -1 & 0 \\
        0 & 0 & -1
    \end{pmatrix}
\end{equation*}
```

- Both rotations are illustrated bellow:

<div style="text-align: center;">
    <img src="./imgs/referential_rotation.jpeg" alt="Treasure map" width="800"/>
</div>

Therefore, the velocities of the robot in the global referential frame are:

```math
\begin{equation*}
    \begin{pmatrix}
        \dot{x} \\
        \dot{y} \\
        \dot{\theta}
    \end{pmatrix} = R_x(\pi)R_z(-\theta)
    \begin{pmatrix}
        v \\
        0 \\
        \omega
    \end{pmatrix} = 
    \begin{pmatrix}
        cos(\theta) & sin(\theta) & 0 \\
        sin(\theta) & -cos(\theta) & 0 \\
        0 & 0 & -1
    \end{pmatrix}
    \begin{pmatrix}
        v \\
        0 \\
        \omega
    \end{pmatrix} =
    \begin{pmatrix}
        vcos(\theta) \\
        vsin(\theta) \\
        -\omega
    \end{pmatrix}
\end{equation*}
```

We can then use these, along with the previous relationship between motor speeds, to define the discrete state-space equations:

```math
\begin{equation*}
    \begin{align*}
        \vec{x}_{k} &= 
            \begin{pmatrix}
                x_{k}\\ 
                y_{k}\\
                \theta_{k}
            \end{pmatrix} 
            = \boldsymbol{g}(\vec{x}_{k-1}, \vec{u}_k) = 
            \begin{pmatrix}
                x_{k-1} + dt \cdot (\frac{r\dot{\phi}_r}{2} + \frac{r\dot{\phi}_l}{2}) \cdot cos(\theta_{k-1})\\ 
                y_{k-1} + dt \cdot (\frac{r\dot{\phi}_r}{2} + \frac{r\dot{\phi}_l}{2}) \cdot sin(\theta_{k-1})\\
                \theta_{k-1} - dt \cdot (\frac{r\dot{\phi}_r}{d} - \frac{r\dot{\phi}_l}{d})
            \end{pmatrix} =
            \begin{pmatrix}
                x_{k-1} + dt \cdot \frac{\lambda}{2} (u_{r,k} + u_{l,k}) \cdot cos(\theta_{k-1})\\ 
                y_{k-1} + dt \cdot \frac{\lambda}{2} (u_{r,k} + u_{l,k}) \cdot sin(\theta_{k-1})\\
                \theta_{k-1} - dt \cdot \frac{\lambda}{d} (u_{r,k} - u_{l,k})
            \end{pmatrix} \quad \text{with} \quad \vec{u}_k =
                \begin{pmatrix}
                    u_{r,k} \\
                    u_{l,k}
                \end{pmatrix} \\
        \vec{z}_k &= 
            \begin{pmatrix}
                x_{k,measured} \\
                y_{k,measured} \\
                \theta_{k,measured}
            \end{pmatrix} = \boldsymbol{h}(\vec{x}_{k}) =
            \begin{pmatrix}
                x_k \\
                y_k \\
                \theta_k
            \end{pmatrix}
    \end{align*}
\end{equation*}
```

Where $dt$ is the sampling period, $\vec{u}_{k}$ is the control input provided by the odometry (i.e., the measured wheel revolutions), and $\lambda$ is a conversion factor that converts motor commands into a linear speed expressed in $mm/s$. 

Notice that the measurement model simply involves directly observing the robot's pose using the camera.

Now that we have defined the state-space representation of our system, we need to estimate the type of noise that will affect it and select an appropriate uncertainty filter algorithm from those discussed in class.

### Filter selection

During the lectures, we studied four different implementations of Bayes filters:

- `Histogram Filter`: can represent any probability distribution, but it requires discretizing the belief into a finite set of bins, meaning the continuous state space must be approximated by a grid. For real-time applications or high-dimensional state spaces, this approach becomes computationally expensive.

- `Particle Filter`: is conceptually similar to the Histogram Filter because it also discretizes the state space, but instead of using a uniform grid, it maintains a set of samples concentrated around the most probable regions of the belief. It is more efficient than the Histogram Filter, but it remains a non-parametric method, so achieving good accuracy often requires a large number of particles, which increases computational cost.

- `Extended Kalman Filter (EKF)`: generalizes the Kalman Filter to nonlinear systems by linearizing the motion and measurement models. This extension sacrifices optimality, and the filter’s accuracy depends strongly on the quality of the linearization. However, because it is still a parametric filter (representing belief as a Gaussian through a mean and covariance), it is computationally efficient. However, it shares the same limitations as the standard Kalman Filter: it assumes Gaussian noise for both the motion and measurement models, as well as a Gaussian initial belief. 

For our project, we use the `Extended Kalman Filter` because the robot’s motion model is nonlinear, yet the assumptions of Gaussian noise and a Gaussian initial belief remain reasonable. In other words, there is no strong counter-indication that would invalidate these assumptions or justify switching to a non-parametric approach such as the Histogram or Particle Filter. 

Moreover, although the state dimension is small, the computational constraints favor a fast parametric estimator like the EKF rather than a more expensive non-parametric method.

### Extended Kalman Filter (EKF)

As described in the [project description document](../docs/project_description.md), the state-space equations are nonlinear due to the projections of the foward and angular velocities of the Thymio. Therefore, we have to use the Extended Kalman Filter to linearized the system at each discrete time k.

Let's consider the previously defined model of the system but with some Gaussian noise incorporated into it:

```math
\begin{equation*}
    \begin{align*}
        \vec{x}_{k} &= 
            \begin{pmatrix}
                x_{k}\\ 
                y_{k}\\
                \theta_{k}
            \end{pmatrix} 
            = \boldsymbol{g}(\vec{x}_{k-1}, \vec{u}_k) = 
            \begin{pmatrix}
                x_{k-1} + dt \cdot \frac{\lambda}{2} (u_{r,k} + u_{l,k}) \cdot cos(\theta_{k-1})\\ 
                y_{k-1} + dt \cdot \frac{\lambda}{2} (u_{r,k} + u_{l,k}) \cdot sin(\theta_{k-1})\\
                \theta_{k-1} - dt \cdot \frac{\lambda}{d} (u_{r,k} - u_{l,k})
            \end{pmatrix} + \epsilon_t \quad \text{with} \quad \vec{u}_k =
                \begin{pmatrix}
                    u_{r,k} \\
                    u_{l,k}
                \end{pmatrix} \quad \text{the odometry.} \\
        \vec{z}_k &= 
            \begin{pmatrix}
                x_{k,measured} \\
                y_{k,measured} \\
                \theta_{k,measured}
            \end{pmatrix} = \boldsymbol{h}(\vec{x}_{k}) =
            \begin{pmatrix}
                x_k \\
                y_k \\
                \theta_k
            \end{pmatrix} + \delta_t
    \end{align*}
\end{equation*}
```

Where $\epsilon_t \sim N(0, Q)$ is a multinormal modelling the uncertainty introduced by the state-space transition, and $\delta_t \sim N(0, R)$ is a multinormal modelling the measurement noise.

As the EKF is an implementation of the general Bayes filter, its algorithm is also decomposed into prediction and update steps. The prediction step is given by the following formula:

```math
\begin{equation*}
    \begin{align*}
        \bar{\mu}_k &= g(\mu_{k-1}, u_k) \\
        \bar{\Sigma}_k &= G_k \Sigma_{k-1} G_k^T + Q_k
    \end{align*}
\end{equation*}
```

And the measurement update step is given by the following:

```math
\begin{equation*}
    \begin{align*}
        K_k &= \bar{\Sigma}_k H_k^T (H_k \bar{\Sigma}_k H_k^T + R_k)^{-1} \\
        \mu_k &= \bar{\mu}_k + K_k (z_k - h(\bar{\mu}_k)) \\
        \Sigma_k &= (I - K_k H_k) \bar{\Sigma}_k
    \end{align*}
\end{equation*}
```

The main difference from the Kalman Filter is that the linear prediction equations are replaced by their nonlinear counterparts, and the linear system matrices are replaced by the corresponding Jacobians.

#### Jacobians matrices

From the previously defined motion model (or state-transition model), we can derive the following Jacobian $G_k$:

```math
\begin{equation}
    \begin{align*}
        G_k &= 
        \begin{pmatrix}
            \frac{\partial g_1}{\partial x_{k-1}} & \frac{\partial g_1}{\partial y_{k-1}} & \frac{\partial g_1}{\partial \theta_{k-1}} \\
            \frac{\partial g_2}{\partial x_{k-1}} & \frac{\partial g_2}{\partial y_{k-1}} & \frac{\partial g_2}{\partial \theta_{k-1}} \\
            \frac{\partial g_3}{\partial x_{k-1}} & \frac{\partial g_3}{\partial y_{k-1}} & \frac{\partial g_3}{\partial \theta_{k-1}}
        \end{pmatrix} \\
        &=
        \begin{pmatrix}
            1 & 0 & -dt \cdot \frac{\lambda}{2} (u_{r,k} + u_{l,k}) \cdot sin(\theta_{k-1}) \\
            0 & 1 & dt \cdot \frac{\lambda}{2} (u_{r,k} + u_{l,k}) \cdot cos(\theta_{k-1}) \\
            0 & 0 & 1
        \end{pmatrix}
    \end{align*}
\end{equation}
```

Similarly, we can derive the following Jacobian, $H_k$, for the measurement model:

```math
\begin{equation}
    \begin{align*}
        H_k &= 
        \begin{pmatrix}
            \frac{\partial h_1}{\partial x_k} & \frac{\partial h_1}{\partial y_k} & \frac{\partial h_1}{\partial \theta_k} \\
            \frac{\partial h_2}{\partial x_k} & \frac{\partial h_2}{\partial y_k} & \frac{\partial h_2}{\partial \theta_k} \\
            \frac{\partial h_3}{\partial x_k} & \frac{\partial h_3}{\partial y_k} & \frac{\partial h_3}{\partial \theta_k}
        \end{pmatrix} \\
        &=
        \begin{pmatrix}
            1 & 0 & 0 \\
            0 & 1 & 0 \\
            0 & 0 & 1
        \end{pmatrix} = I_{3x3}
    \end{align*}
\end{equation}
```

#### Noise estimation

Finally, to implement the EKF, we must estimate the noise characteristics $Q_k$ and $R_k$​, which capture the robot’s motion uncertainty and sensor limitations. The more accurately these covariance matrices reflect the real system behavior, the more effective and reliable the EKF becomes.

1. We need to estimate the covariance $Q_k$ of the stochastic noise $\epsilon_t \sim N(0, Q)$ affecting the state-transition model.

```math
\begin{equation}
    Q_k = 
    \begin{pmatrix}
        q_x & 0 & 0 \\
        0 & q_y & 0 \\
        0 & 0 & q_\theta
    \end{pmatrix} = ?
\end{equation}
```

2. We need to estimate the covariance $R_k$ of the stochastic noise $\delta_t \sim N(0, R)$ describing the measurement noise.

```math
\begin{equation}
    R_k =
    \begin{pmatrix}
        r_x & 0 & 0 \\
        0 & r_y & 0 \\
        0 & 0 & r_\theta
    \end{pmatrix} = ?
\end{equation}
```


**TODO:**
- Add the feature that there is no update step when the camera is obstructed / not detect the robot

- Compute $Q$ and $R$

- At each main iteration, just before motion control, call Kalman and use the $\mu$ for the control